[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/02_mathematics_for_ml/90_exercises.ipynb)

# Bài tập — Chương 02: Toán cho ML (L0 → L7)

**Hướng dẫn:** câu lý thuyết trả lời vào cell "✍️ Trả lời"; câu tính tay/code điền vào `### START CODE HERE ###` … `### END CODE HERE ###` rồi chạy cell kiểm tra.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/02_mathematics_for_ml

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from ml_course.checks import check_close, check_equal, check_less, check_true
from ml_course.utils import numerical_gradient, relative_error

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Định nghĩa: (a) hạng của ma trận; (b) vector riêng; (c) gradient; (d) likelihood.

✍️ Trả lời:

**Đáp án:** (a) số cột (hàng) độc lập tuyến tính tối đa; (b) vector $\mathbf v \ne 0$ thoả $A\mathbf v = \lambda\mathbf v$ — chỉ bị co giãn bởi $A$; (c) vector các đạo hàm riêng, chỉ hướng tăng nhanh nhất của hàm; (d) xác suất (mật độ) của dữ liệu đã quan sát, xem như hàm của tham số.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích vì sao cực tiểu MSE tương đương với cực đại likelihood, và giả định nào làm điều đó đúng.

✍️ Trả lời:

**Đáp án:** Nếu $y = \mathbf w^\top\mathbf x + \varepsilon$ với nhiễu $\varepsilon \sim \mathcal N(0, \sigma^2)$ độc lập cùng phân phối, thì $-\log\mathcal L(\mathbf w) = \frac{1}{2\sigma^2}\sum_i(y_i - \mathbf w^\top\mathbf x_i)^2 + \text{const}$. Cực đại likelihood ⇔ cực tiểu tổng bình phương sai số. Giả định then chốt: nhiễu Gauss, độc lập, phương sai không đổi.

**Câu 3.** Vì sao chuẩn hoá feature giúp gradient descent hội tụ nhanh hơn? Liên hệ với condition number.

✍️ Trả lời:

**Đáp án:** Feature có thang đo chênh lệch → loss surface là elip rất dẹt (Hessian có condition number lớn). Learning rate bị giới hạn bởi hướng cong nhất, nên theo hướng còn lại GD đi rất chậm và zig-zag. Chuẩn hoá đưa các feature về cùng thang đo → đường đồng mức "tròn" hơn → condition number nhỏ → hội tụ nhanh.

## L2 — Tính tay

**Câu 4.** Tính tay rồi điền kết quả:
- (a) $\lVert[3, -4]\rVert_2$ và $\lVert[3, -4]\rVert_1$
- (b) cosine similarity giữa $[1, 0]$ và $[1, 1]$
- (c) $C(6, 2)$ và $P(6, 2)$
- (d) xét nghiệm: tỉ lệ bệnh 2%, độ nhạy 90%, dương tính giả 10% → $P(\text{bệnh} \mid +)$
- (e) gradient của $f(x, y) = x^2 y + 3y$ tại $(2, 1)$

In [ ]:
### START CODE HERE ### (≈ 7 dòng)
l2_norm, l1_norm = 5.0, 7.0
cos_sim = 1 / math.sqrt(2)
comb_6_2, perm_6_2 = 15, 30
p_sick_given_pos = 0.9 * 0.02 / (0.9 * 0.02 + 0.1 * 0.98)
grad_at_2_1 = [4.0, 7.0]
### END CODE HERE ###

In [ ]:
check_close((l2_norm, l1_norm), (np.linalg.norm([3, -4]), np.linalg.norm([3, -4], 1)), message="(a) chuẩn L2, L1")
check_close(cos_sim, 1 / np.sqrt(2), message="(b) cosine")
check_close((comb_6_2, perm_6_2), (math.comb(6, 2), math.perm(6, 2)), message="(c) tổ hợp, chỉnh hợp")
check_close(p_sick_given_pos, 0.15517, tol=1e-4, message="(d) Bayes")
f = lambda v: v[0] ** 2 * v[1] + 3 * v[1]
check_close(grad_at_2_1, numerical_gradient(f, np.array([2.0, 1.0])), tol=1e-5, message="(e) gradient")

## L3 — Code

**Bài 5.** Cài đặt gradient descent tổng quát `gd(grad, x0, lr, n_iters)` trả về điểm cuối, rồi dùng nó cực tiểu hàm Rosenbrock thu nhỏ $f(x, y) = (1 - x)^2 + 5(y - x^2)^2$ (cực tiểu tại $(1, 1)$).

In [ ]:
def gd(grad, x0, lr, n_iters):
    x = np.asarray(x0, dtype=float)
    ### START CODE HERE ### (≈ 2 dòng)
    for _ in range(n_iters):
        x = x - lr * grad(x)
    ### END CODE HERE ###
    return x


def rosen_grad(v):
    x, y = v
    ### START CODE HERE ### (≈ 2 dòng)
    gx = -2 * (1 - x) - 20 * x * (y - x**2)
    gy = 10 * (y - x**2)
    ### END CODE HERE ###
    return np.array([gx, gy])


x_star = gd(rosen_grad, [-1.0, 1.5], lr=0.01, n_iters=20_000)
print(x_star)

In [ ]:
rosen = lambda v: (1 - v[0]) ** 2 + 5 * (v[1] - v[0] ** 2) ** 2
check_less(relative_error(rosen_grad(np.array([0.3, -0.7])), numerical_gradient(rosen, np.array([0.3, -0.7]))), 1e-6,
           "gradient đúng (gradient check)")
check_close(x_star, [1, 1], tol=1e-3, message="GD tìm được cực tiểu (1, 1)")

**Bài 6.** Viết `mle_poisson(counts)` trả về $\hat\lambda_{MLE}$ của phân phối Poisson (gợi ý: đạo hàm log-likelihood $\sum_i [k_i\log\lambda - \lambda]$ theo $\lambda$ rồi cho bằng 0), và `bootstrap_mean_ci(x, n_boot, seed)` trả về khoảng tin cậy 95% của trung bình bằng bootstrap (from scratch, không dùng `ml_course.validation`).

In [ ]:
def mle_poisson(counts):
    ### START CODE HERE ### (≈ 1 dòng)
    lam = np.mean(counts)
    ### END CODE HERE ###
    return lam


def bootstrap_mean_ci(x, n_boot=5000, seed=0):
    r = np.random.default_rng(seed)
    x = np.asarray(x)
    ### START CODE HERE ### (≈ 3 dòng)
    idx = r.integers(0, len(x), size=(n_boot, len(x)))
    boot_means = x[idx].mean(axis=1)
    ci = (np.quantile(boot_means, 0.025), np.quantile(boot_means, 0.975))
    ### END CODE HERE ###
    return ci

In [ ]:
calls = rng.poisson(3.2, size=500)
check_close(mle_poisson(calls), calls.mean(), message="λ_MLE = trung bình mẫu")
lo, hi = bootstrap_mean_ci(calls)
check_true(lo < calls.mean() < hi and hi - lo < 0.5, "CI chứa trung bình mẫu và đủ hẹp")

## L4 — Debug

**Bài 7.** Đồng nghiệp viết gradient của MSE cho linear regression nhưng model không hội tụ. Dùng **gradient check** để tìm và sửa lỗi.

In [ ]:
Xd, yd = rng.normal(size=(50, 3)), rng.normal(size=50)


def mse_loss(w):
    return np.mean((Xd @ w - yd) ** 2)


def mse_grad(w):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return 2 * Xd @ (Xd.T @ w - yd) / len(yd)     # 🐛
    return 2 * Xd.T @ (Xd @ w - yd) / len(yd)
    ### END CODE HERE ###


w0 = rng.normal(size=3)
print("relative error:", relative_error(mse_grad(w0), numerical_gradient(mse_loss, w0.copy())))

In [ ]:
check_less(relative_error(mse_grad(w0), numerical_gradient(mse_loss, w0.copy())), 1e-7, "gradient MSE đúng")

**Giải thích:** công thức đúng là $\frac{2}{n}X^\top(X\mathbf w - \mathbf y)$. Bản lỗi đảo vị trí $X$ và $X^\top$ — vẫn chạy được trong một số trường hợp shape tình cờ khớp, nên **chỉ gradient check mới phát hiện được**.

## L5 — Phân tích

**Bài 8.** Mô phỏng: trung bình mẫu của phân phối **mũ** (exponential, lệch phải) với $n \in \{2, 10, 50\}$. Với mỗi $n$, lặp 20 000 lần, vẽ histogram của trung bình mẫu và tính **độ lệch chuẩn** của chúng. Kiểm tra: độ lệch chuẩn có tỉ lệ với $1/\sqrt n$ không? Hình dạng phân phối thay đổi ra sao?

In [ ]:
ns = [2, 10, 50]
stds = []
fig, ax = plt.subplots()
### START CODE HERE ### (≈ 4 dòng)
for n in ns:
    means = rng.exponential(1.0, size=(20_000, n)).mean(axis=1)
    stds.append(means.std())
    ax.hist(means, bins=80, density=True, histtype="step", label=f"n={n}")
### END CODE HERE ###
ax.legend()
plt.show()
print("std:", np.round(stds, 4), "| 1/sqrt(n):", np.round(1 / np.sqrt(ns), 4))

In [ ]:
check_close(stds, 1 / np.sqrt(ns), tol=0.03, message="std của trung bình ≈ σ/√n (σ = 1)")

✍️ Nhận xét:

**Gợi ý:** độ lệch chuẩn ≈ $\sigma/\sqrt n$ (σ = 1 với Exp(1)) — đúng như CLT. Với $n = 2$ phân phối còn lệch phải rõ; $n = 50$ đã gần hình chuông. Hệ quả cho ML: điểm số trên tập test nhỏ dao động mạnh; tăng gấp 4 kích thước test chỉ giảm sai số chuẩn còn một nửa.

## L6 — Thực tế

**Bài 9.** Hai phiên bản giao diện đặt hàng được A/B test: A có 4 800 khách, 432 người mua; B có 5 100 khách, 510 người mua. Tính tỉ lệ chuyển đổi mỗi bản, **khoảng tin cậy 95% bootstrap cho hiệu số** (B − A), và kết luận có nên triển khai B không.

In [ ]:
a_conv = np.r_[np.ones(432), np.zeros(4800 - 432)]
b_conv = np.r_[np.ones(510), np.zeros(5100 - 510)]
### START CODE HERE ### (≈ 5 dòng)
diffs = []
for _ in range(3000):
    diffs.append(rng.choice(b_conv, len(b_conv)).mean() - rng.choice(a_conv, len(a_conv)).mean())
ci_low, ci_high = np.quantile(diffs, [0.025, 0.975])
uplift = b_conv.mean() - a_conv.mean()
### END CODE HERE ###
print(f"A: {a_conv.mean():.3%}  B: {b_conv.mean():.3%}  uplift = {uplift:.3%}  CI95 = [{ci_low:.3%}, {ci_high:.3%}]")

In [ ]:
check_close(uplift, 510 / 5100 - 432 / 4800, message="uplift")
check_true(ci_low < uplift < ci_high, "CI chứa uplift quan sát")

✍️ Kết luận:

**Gợi ý:** uplift ≈ 1 điểm phần trăm, CI 95% xấp xỉ [−0.2%, 2.2%] — **chứa 0** → chưa đủ bằng chứng thống kê. Nên chạy test lâu hơn (thêm dữ liệu) hoặc cân nhắc chi phí triển khai trước khi quyết định (chương 40 — A/B testing).

## L7 — Nâng cao

**Bài 10.** Cài đặt **PCA from scratch bằng power iteration** với *deflation*: tìm 2 vector riêng lớn nhất của ma trận hiệp phương sai của bộ dữ liệu `digits` (64 chiều). Sau khi tìm vector riêng thứ nhất $\mathbf v_1$ với trị riêng $\lambda_1$, "loại bỏ" nó: $C \leftarrow C - \lambda_1\mathbf v_1\mathbf v_1^\top$ rồi lặp lại. So sánh với `np.linalg.eigh`.

In [ ]:
from ml_course.datasets import load_digits_data

Xg, yg, _ = load_digits_data()
C = np.cov(Xg.T)


def top_k_eigen(C, k=2, n_iters=500, seed=0):
    r = np.random.default_rng(seed)
    C = C.copy()
    vals, vecs = [], []
    for _ in range(k):
        v = r.normal(size=C.shape[0])
        ### START CODE HERE ### (≈ 6 dòng)
        for _ in range(n_iters):
            v = C @ v
            v /= np.linalg.norm(v)
        lam = v @ C @ v
        C = C - lam * np.outer(v, v)
        vals.append(lam)
        vecs.append(v)
        ### END CODE HERE ###
    return np.array(vals), np.array(vecs)


vals, vecs = top_k_eigen(C)
ref_vals, ref_vecs = np.linalg.eigh(np.cov(Xg.T))
print("power iteration:", vals.round(4), "| eigh:", ref_vals[::-1][:2].round(4))

In [ ]:
check_close(vals, ref_vals[::-1][:2], tol=1e-4, message="2 trị riêng lớn nhất")
check_close(np.abs(vecs @ ref_vecs[:, ::-1][:, :2]).diagonal(), [1, 1], tol=1e-3, message="vector riêng trùng (bỏ qua dấu)")

In [ ]:
Z = (Xg - Xg.mean(0)) @ vecs.T
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=yg, cmap="tab10", s=6)
fig.colorbar(sc)
ax.set_title("digits chiếu lên 2 thành phần chính (PCA from scratch)")
plt.show()